# GRU vs LSTM comparison

The reported results in the manuscript come from the existing
`MultiStreamLSTM` class in `baseline_lstm.py`, which actually uses
`nn.GRU` internally despite its name. This notebook trains a true LSTM
variant on the same fold (All streams, shared label policy, 2020 test
year) and reports both architectures side by side, so we can either
confirm the choice of GRU or switch to LSTM if it performs better.

We also save the per-day logits from the GRU model so that figure F8
(example LSTM probability curves) can be regenerated without rerunning
the F8 add-on separately.

Total runtime: roughly 20 to 40 minutes on the MIG A100 slice (two
fold trainings plus two inference passes).


## 1. Setup

In [15]:
from pathlib import Path
import sys, time, gc
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path('/fs1/home/atikul/Phenology')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from phase3_baselines.dataset import (load_year_bundle, SelectionPolicy,
                                       apply_selection, concat_bundles)
from phase3_baselines.baseline_gru_lstm_compare import (
    StreamEncoderGRU, StreamEncoderLSTM,
    train_one_fold_generic, collect_logits,
)
from phase3_baselines.evaluate import evaluate_predictions, print_evaluation

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

TENSORS_DIR = PROJECT_ROOT / 'cache' / 'tensors'
LSTM_DIR = PROJECT_ROOT / 'phase3_outputs' / 'lstm'
LSTM_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = PROJECT_ROOT / 'phase3_outputs' / 'gru_vs_lstm'
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2017, 2018, 2019, 2020, 2021]
TEST_YEAR = 2020
ACTIVE_STREAMS = ('S1', 'S2', 'Met')

# === Training hyperparameters ===
# Use the codebase defaults that produced the existing lstm_results.csv numbers.
# If you trained the existing results with different values, edit here to match.
HIDDEN = 64
NUM_LAYERS = 1
EPOCHS = 12
BATCH_SIZE = 64
LR = 1e-3
WD = 1e-5


Device: cuda


## 2. Load the 2020 LOYO fold

In [16]:
print('Loading bundles...')
policy = SelectionPolicy(use_only_rank1=False)
test_b = apply_selection(
    load_year_bundle(TENSORS_DIR / f'phenology_{TEST_YEAR}.npz'), policy)
train_parts = {}
for y in YEARS:
    if y == TEST_YEAR:
        continue
    train_parts[y] = apply_selection(
        load_year_bundle(TENSORS_DIR / f'phenology_{y}.npz'), policy)
train_b = concat_bundles(train_parts, years=[y for y in YEARS if y != TEST_YEAR])
del train_parts
gc.collect()
print(f'  train: {train_b["X"].shape}, test: {test_b["X"].shape}')

PHASE_IDS = train_b.get('phase_ids', [10, 12, 15, 18, 19, 21, 24])


Loading bundles...
  train: (21634, 426, 19), test: (5181, 426, 19)


## 3. Train the GRU variant

In [17]:
print('=== GRU encoder ===')
torch.manual_seed(0)  # reproducibility within the seed-sensitive parts we control
t0 = time.time()
gru_result = train_one_fold_generic(
    train_b=train_b, test_b=test_b,
    active=ACTIVE_STREAMS,
    encoder_cls=StreamEncoderGRU,
    hidden=HIDDEN, num_layers=NUM_LAYERS,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    lr=LR, weight_decay=WD,
    device=DEVICE, verbose=True,
    return_model=True,    # keep the model in scope so we can collect logits
)
print(f'GRU fold done in {time.time()-t0:.0f}s')


=== GRU encoder ===
    epoch  1/12: train_loss=0.1016
    epoch  2/12: train_loss=0.0645
    epoch  3/12: train_loss=0.0621
    epoch  4/12: train_loss=0.0611
    epoch  5/12: train_loss=0.0606
    epoch  6/12: train_loss=0.0601
    epoch  7/12: train_loss=0.0597
    epoch  8/12: train_loss=0.0597
    epoch  9/12: train_loss=0.0595
    epoch 10/12: train_loss=0.0594
    epoch 11/12: train_loss=0.0592
    epoch 12/12: train_loss=0.0592
GRU fold done in 121s


## 4. Train the LSTM variant

In [18]:
print('=== LSTM encoder ===')
torch.manual_seed(0)
t0 = time.time()
lstm_result = train_one_fold_generic(
    train_b=train_b, test_b=test_b,
    active=ACTIVE_STREAMS,
    encoder_cls=StreamEncoderLSTM,
    hidden=HIDDEN, num_layers=NUM_LAYERS,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    lr=LR, weight_decay=WD,
    device=DEVICE, verbose=True,
    return_model=True,
)
print(f'LSTM fold done in {time.time()-t0:.0f}s')


=== LSTM encoder ===
    epoch  1/12: train_loss=0.1067
    epoch  2/12: train_loss=0.0645
    epoch  3/12: train_loss=0.0622
    epoch  4/12: train_loss=0.0611
    epoch  5/12: train_loss=0.0608
    epoch  6/12: train_loss=0.0604
    epoch  7/12: train_loss=0.0600
    epoch  8/12: train_loss=0.0598
    epoch  9/12: train_loss=0.0596
    epoch 10/12: train_loss=0.0595
    epoch 11/12: train_loss=0.0594
    epoch 12/12: train_loss=0.0591
LSTM fold done in 121s


## 5. Side-by-side evaluation

In [19]:
report_gru  = evaluate_predictions(
    pred_days=gru_result['pred'].astype(np.float32),
    true_days=gru_result['truth'].astype(np.float32),
    phase_ids=PHASE_IDS,
    years=gru_result['year'],
    ranks=gru_result['rank'],
)
report_lstm = evaluate_predictions(
    pred_days=lstm_result['pred'].astype(np.float32),
    true_days=lstm_result['truth'].astype(np.float32),
    phase_ids=PHASE_IDS,
    years=lstm_result['year'],
    ranks=lstm_result['rank'],
)

print('=== GRU (existing architecture) ===')
print_evaluation(report_gru, 'GRU')
print()
print('=== LSTM (new variant) ===')
print_evaluation(report_lstm, 'LSTM')


=== GRU (existing architecture) ===

GRU

[Headline — macro over phases]
  macro_mae               : 8.321
  macro_r2                : -0.031
  macro_within_5d         : 0.440
  n_phases_evaluated      : 7

[Per phase]
          n     mae    rmse     r2  within_5d   bias
phase                                               
10     4719  12.008  14.755 -0.462      0.271 -8.080
12     4628  10.861  13.608 -0.053      0.326 -4.293
15     4321   8.563  10.643 -0.007      0.387 -1.024
18     4758   5.875   7.842  0.081      0.581  0.476
19     4170   8.632  11.174 -0.157      0.414 -5.118
21     4422   6.998   9.128  0.160      0.494 -1.976
24     4986   5.313   7.118  0.218      0.608 -1.889

[Per (phase, year)]
year    2020
phase       
10     12.01
12     10.86
15      8.56
18      5.87
19      8.63
21      7.00
24      5.31

[Per (phase, rank) — MAE]
rank      1      2      3      4      5      6      7      8      9      10
phase                                                          

In [20]:
# Side-by-side per-phase table
g_per = report_gru['per_phase'][['mae','rmse','within_5d']].round(2)
l_per = report_lstm['per_phase'][['mae','rmse','within_5d']].round(2)
g_per.columns = [f'GRU_{c}' for c in g_per.columns]
l_per.columns = [f'LSTM_{c}' for c in l_per.columns]
combined = g_per.join(l_per)
combined['MAE_diff_LSTM_minus_GRU'] = (combined['LSTM_mae'] - combined['GRU_mae']).round(2)
print('=== Per-phase comparison (2020 fold, All streams, shared label) ===')
print(combined.to_string())
combined.to_csv(OUT_DIR / 'gru_vs_lstm_per_phase_2020.csv')
print(f'\nSaved {OUT_DIR / "gru_vs_lstm_per_phase_2020.csv"}')


=== Per-phase comparison (2020 fold, All streams, shared label) ===
       GRU_mae  GRU_rmse  GRU_within_5d  LSTM_mae  LSTM_rmse  LSTM_within_5d  MAE_diff_LSTM_minus_GRU
phase                                                                                                
10       12.01     14.76           0.27     10.29      12.90            0.35                    -1.72
12       10.86     13.61           0.33     10.71      13.53            0.35                    -0.15
15        8.56     10.64           0.39      8.71      10.95            0.39                     0.15
18        5.87      7.84           0.58      6.60       8.67            0.53                     0.73
19        8.63     11.17           0.41      7.52       9.77            0.47                    -1.11
21        7.00      9.13           0.49      6.88       9.16            0.50                    -0.12
24        5.31      7.12           0.61      5.38       7.18            0.61                     0.07

Saved /fs1/ho

In [21]:
# Macro summary
def macro(rep):
    return {'macro_mae': rep['per_phase']['mae'].mean(),
            'macro_rmse': rep['per_phase']['rmse'].mean(),
            'macro_within_5d': rep['per_phase']['within_5d'].mean()}
summary = pd.DataFrame({
    'GRU':  macro(report_gru),
    'LSTM': macro(report_lstm),
}).T
summary['MAE_diff_LSTM_minus_GRU'] = (summary['macro_mae'] - summary.loc['GRU','macro_mae']).round(3)
print('=== Macro summary (2020 fold) ===')
print(summary.round(3).to_string())
summary.to_csv(OUT_DIR / 'gru_vs_lstm_macro_2020.csv')


=== Macro summary (2020 fold) ===
      macro_mae  macro_rmse  macro_within_5d  MAE_diff_LSTM_minus_GRU
GRU       8.321      10.610            0.440                    0.000
LSTM      8.013      10.308            0.459                   -0.308


## 6. Save per-day logits for F8

The figure F8 (example LSTM probability curves) needs the full per-day
phase probability surfaces. We save them here from whichever model the
manuscript ends up reporting. Default is GRU (the existing architecture
that produced the headline results); change `MODEL_FOR_F8` if you
decide the manuscript should report the LSTM variant instead.


In [ ]:
MODEL_FOR_F8 = 'GRU'   # change to 'LSTM' if you decide to switch

if MODEL_FOR_F8 == 'GRU':
    model = gru_result['model']
    loader = gru_result['test_loader']
elif MODEL_FOR_F8 == 'LSTM':
    model = lstm_result['model']
    loader = lstm_result['test_loader']
else:
    raise ValueError(f'unknown MODEL_FOR_F8={MODEL_FOR_F8}')

inf = collect_logits(model, loader, active=ACTIVE_STREAMS, device=DEVICE)
print(f'logits: {inf["logits"].shape}')

NPZ = LSTM_DIR / 'lstm_logits.npz'    # filename kept for fig8 compatibility
pref = f'All__all10__{TEST_YEAR}'
arrays = {
    f'{pref}__logits': inf['logits'].astype(np.float32),
    f'{pref}__yt':     inf['yt'],
    f'{pref}__xid':    inf['xid'],
    f'{pref}__bag':    inf['bag'],
    f'{pref}__rank':   inf['rank'],
    f'{pref}__year':   inf['year'],
}

# Merge with existing file if present (preserves logits from other folds if any)
existing = {}
if NPZ.exists():
    with np.load(NPZ, allow_pickle=False) as zz:
        existing = {k: zz[k] for k in zz.files}
    print(f'Merging with {len(existing)} existing arrays')
existing.update(arrays)
np.savez_compressed(NPZ, **existing)
print(f'Saved -> {NPZ}')
print('\nNow re-run notebook 09_paper_figures_v2.ipynb and F8 will be generated.')


## 7. What this tells us, and what to do next

Look at the **macro_mae_diff_LSTM_minus_GRU** number in section 5:

- **|diff| < 0.2 days**: GRU and LSTM are within noise of each other.
  The current paper framing (GRU model, justification: parameter
  efficiency) is fine. Keep the manuscript saying GRU and add a one
  paragraph note that LSTM was tested and produced comparable results.

- **diff > +0.3 days** (LSTM worse): LSTM underperforms, presumably from
  the extra parameter count over the same training set. Keep GRU as the
  primary model, mention LSTM was tested and was slightly worse, frame
  the GRU choice as deliberate.

- **diff < −0.3 days** (LSTM better): LSTM is genuinely better. Worth
  considering a switch. In that case you would want to rerun the full
  5-fold cross-validation for the LSTM variant before switching the
  paper's headline numbers, since one fold could be a fluke.

In all three cases the manuscript should be updated to use the word
"GRU" instead of "LSTM" where the current paper draft uses it, since
that is what produced the existing results.
